# MTP speculative decoding in Rapid-LLM

Open this notebook in a **Google Colab CPU runtime** with the project workspace available. Set `WORKSPACE` in the first code cell to the directory containing the modified `Rapid-LLM/` folder. The notebook runs that source directly and installs the analytical simulator dependencies into an isolated virtual environment.

Run the cells in order to compare ordinary decoding and three-guess MTP and verify that explicitly setting zero guesses reproduces ordinary decode.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Set this to the workspace folder if Colab starts in a different directory.
WORKSPACE = Path.cwd().resolve()
REPO = WORKSPACE / "Rapid-LLM"

model_path = REPO / "configs/model-config/MiMo-7B_mtp.yaml"
if not (REPO / "run_perf.py").is_file() or not model_path.is_file():
    raise FileNotFoundError(
        f"Expected the modified Rapid-LLM project at {REPO}. "
        "Set WORKSPACE to the folder containing Rapid-LLM."
    )

# Keep the virtual environment and generated reports on Colab's local filesystem.
ROOT = Path("/content/rapid-llm-mtp").resolve()
ROOT.mkdir(parents=True, exist_ok=True)
print("Using local project source:", REPO)


In [ ]:
# Install into an isolated Linux virtual environment, outside the Colab kernel.
if sys.version_info < (3, 11):
    raise RuntimeError("Rapid-LLM requires Python 3.11 or later.")

ENV = ROOT / ".venv"
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)

requirements = ROOT / "requirements-colab.txt"
requirements.write_text("annotated-types==0.7.0\n    # via pydantic\nblinker==1.9.0\n    # via flask\ncertifi==2026.2.25\n    # via requests\ncharset-normalizer==3.4.7\n    # via requests\nclick==8.1.8\n    # via\n    #   flask\n    #   rapid-llm\ncolorama==0.4.6 ; sys_platform == 'win32'\n    # via\n    #   click\n    #   pytest\n    #   tqdm\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\ndash==4.1.0\n    # via\n    #   dash-ag-grid\n    #   dash-mantine-components\n    #   rapid-llm\ndash-ag-grid==35.2.0\n    # via rapid-llm\ndash-iconify==0.1.2\n    # via rapid-llm\ndash-mantine-components==2.6.1\n    # via rapid-llm\nflask==3.1.3\n    # via dash\nfonttools==4.60.1\n    # via matplotlib\ngraphviz==0.21\n    # via rapid-llm\ngreenlet==3.4.0\n    # via playwright\nidna==3.11\n    # via requests\nimportlib-metadata==9.0.0\n    # via dash\niniconfig==2.3.0\n    # via pytest\nitsdangerous==2.2.0\n    # via flask\njinja2==3.1.6\n    # via flask\njoblib==1.5.2\n    # via rapid-llm\nkiwisolver==1.4.9\n    # via matplotlib\nmarkupsafe==3.0.3\n    # via\n    #   flask\n    #   jinja2\n    #   werkzeug\nmatplotlib==3.10.7\n    # via\n    #   rapid-llm\n    #   seaborn\nnarwhals==2.19.0\n    # via plotly\nnest-asyncio==1.6.0\n    # via dash\nnumpy==2.3.3\n    # via\n    #   contourpy\n    #   matplotlib\n    #   pandas\n    #   rapid-llm\n    #   seaborn\npackaging==25.0\n    # via\n    #   matplotlib\n    #   plotly\n    #   pytest\npandas==2.3.1\n    # via\n    #   rapid-llm\n    #   seaborn\npillow==12.0.0\n    # via matplotlib\nplaywright==1.58.0\n    # via rapid-llm\nplotly==6.7.0\n    # via\n    #   dash\n    #   rapid-llm\npluggy==1.6.0\n    # via pytest\nprotobuf==5.28.2\n    # via rapid-llm\npsutil==7.2.2\n    # via rapid-llm\npydantic==2.12.5\n    # via rapid-llm\npydantic-core==2.41.5\n    # via pydantic\npydot==4.0.1\n    # via rapid-llm\npyee==13.0.1\n    # via playwright\npygments==2.19.2\n    # via pytest\npyparsing==3.2.5\n    # via\n    #   matplotlib\n    #   pydot\npytest==9.0.0\n    # via rapid-llm\npython-dateutil==2.9.0.post0\n    # via\n    #   matplotlib\n    #   pandas\n    #   rapid-llm\npytz==2025.2\n    # via\n    #   pandas\n    #   rapid-llm\npyyaml==6.0.2\n    # via rapid-llm\nrequests==2.33.1\n    # via dash\nretrying==1.4.2\n    # via dash\n    # via rapid-llm\nseaborn==0.13.2\n    # via rapid-llm\nsetuptools==82.0.1\n    # via dash\nsix==1.17.0\n    # via\n    #   python-dateutil\n    #   rapid-llm\ntqdm==4.67.1\n    # via rapid-llm\ntyping-extensions==4.15.0\n    # via\n    #   dash\n    #   pydantic\n    #   pydantic-core\n    #   pyee\n    #   typing-inspection\ntyping-inspection==0.4.2\n    # via pydantic\ntzdata==2025.2\n    # via\n    #   pandas\n    #   rapid-llm\nurllib3==2.6.3\n    # via requests\nwerkzeug==3.1.8\n    # via\n    #   dash\n    #   flask\nzipp==3.23.0\n    # via importlib-metadata\n", encoding="utf-8")
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(requirements)], check=True)

In [ ]:
import json
import re

OUTPUT = ROOT / "results"
CONFIGS = ROOT / "configs"
CONFIGS.mkdir(exist_ok=True)

hardware = REPO / "configs/hardware-config/a100_80GB_no_parallelism.yaml"
model = REPO / "configs/model-config/MiMo-7B_mtp.yaml"

# Parse and write YAML using PyYAML in the project virtual environment.
subprocess.run(
    [str(PYTHON), "-c", """
from pathlib import Path
import sys
import yaml
source, target = map(Path, sys.argv[1:])
config = yaml.safe_load(source.read_text(encoding='utf-8'))
config['inference_param'].pop('mtp')
(target / 'baseline.yaml').write_text(yaml.safe_dump(config), encoding='utf-8')
config['inference_param']['mtp'] = {'guesses': 0}
(target / 'zero.yaml').write_text(yaml.safe_dump(config), encoding='utf-8')
""", str(model), str(CONFIGS)],
    check=True,
)

def simulate(name, model_config):
    output = OUTPUT / name
    subprocess.run(
        [str(PYTHON), "-X", "utf8", "run_perf.py",
         "--hardware_config", str(hardware), "--model_config", str(model_config),
         "--output_dir", str(output)],
        cwd=REPO,
        check=True,
    )
    return output / "LLM"

reports = {
    "Baseline": simulate("baseline", CONFIGS / "baseline.yaml"),
    "Explicit D = 0": simulate("zero", CONFIGS / "zero.yaml"),
    "Three MTP guesses": simulate("mtp", model),
}

In [ ]:
from IPython.display import Markdown, display

timing_name = "LLM_inference_results.txt"
memory_name = "memory-summary/memory_capacity_comparison.txt"
for relative in (timing_name, memory_name):
    if (reports["Baseline"] / relative).read_bytes() != (reports["Explicit D = 0"] / relative).read_bytes():
        raise AssertionError(f"Zero-guess baseline mismatch in {relative}")
print("Verified: the baseline and explicit D = 0 timing and memory reports match.")

def metric(report, label):
    return float(re.search(rf"^{re.escape(label)}: ([0-9.]+)", report, re.MULTILINE)[1])

# The baseline report displays total decode time. This workload generates 1,024 tokens.
baseline_report = (reports["Baseline"] / timing_name).read_text(encoding="utf-8")
mtp_report = (reports["Three MTP guesses"] / timing_name).read_text(encoding="utf-8")
baseline_ms = metric(baseline_report, "Decode Time") * 1000 / 1024
baseline_memory = metric(
    (reports["Baseline"] / memory_name).read_text(encoding="utf-8"),
    "Max peak memory usage (per gpu)",
)
mtp_ms = metric(mtp_report, "Mean decode latency per output token")
mtp_memory = metric(mtp_report, "Peak memory per GPU")

display(Markdown(f"""
| Configuration | Decode latency (ms/token) | Peak GPU memory (GiB) |
| --- | ---: | ---: |
| Baseline | {baseline_ms:.2f} | {baseline_memory:.2f} |
| Three MTP guesses | {mtp_ms:.2f} | {mtp_memory:.2f} |

**Modeled decode speedup:** {baseline_ms / mtp_ms:.2f}×  
**Reports:** `{OUTPUT}`
"""))